##### Yasamin Tari, SA
##### 05/12/2026
# Inference Table Migration: AI Gateway V1 → Unity AI Gateway V2

This notebook migrates data from **AI Gateway V1** inference tables to the new **Unity AI Gateway V2** schema.

## Background

Databricks has evolved inference tables through multiple generations:

1. **Legacy Inference Tables** - Original implementation (being deprecated April 2026)
2. **AI Gateway V1 Inference Tables** - First AI Gateway implementation
3. **Unity AI Gateway V2** - Current architecture with Shinkansen ingestion, faster logging (<5 min)

This notebook handles migration from **V1 → V2**.


This notebook uses `MERGE INTO` instead of `APPEND` for **idempotent migration**:

| Approach | Behavior | Use Case |
|----------|----------|----------|
| `APPEND` | Always inserts all records | One-time migration, creates duplicates if re-run |
| `MERGE INTO` | Only inserts non-matching records | Safe to re-run, no duplicates, production-ready |

With `MERGE INTO`, you can safely re-run this notebook multiple times without creating duplicate records.

## Schema Comparison

### AI Gateway V1 Schema (Source)

| Column | Type | Description |
|--------|------|-------------|
| `databricks_request_id` | STRING | Databricks-generated request ID |
| `client_request_id` | STRING | Client-provided request ID |
| `request_date` | DATE | UTC date of request |
| `request_time` | TIMESTAMP | When request was received |
| `status_code` | INT | HTTP status code |
| `sampling_fraction` | DOUBLE | Sampling fraction if downsampled |
| `execution_duration_ms` | BIGINT | Model inference time (ms) |
| `request` | STRING | Raw request JSON |
| `response` | STRING | Raw response JSON |
| `served_entity_id` | STRING | ID of served entity |
| `logging_error_codes` | ARRAY<STRING> | Errors during logging |
| `requester` | STRING | User/SP who made request |

### Unity AI Gateway V2 Schema (Target)

| Column | Type | Description |
|--------|------|-------------|
| `event_time` | TIMESTAMP | When request was received |
| `request_id` | STRING | Request identifier |
| `request_tags` | MAP<STRING,STRING> | Metadata tags (includes client_request_id) |
| `status_code` | INT | HTTP status code |
| `sampling_fraction` | DOUBLE | Sampling fraction if downsampled |
| `latency_ms` | BIGINT | Total latency (ms) |
| `request` | STRING | Raw request JSON |
| `response` | STRING | Raw response JSON |
| `destination_id` | STRING | ID of destination/served entity |
| `logging_error_codes` | ARRAY<STRING> | Errors during logging |
| `requester` | STRING | User/SP who made request |
| `time_to_first_byte_ms` | BIGINT | Time to first response byte (new) |
| `schema_version` | STRING | Schema version identifier (new) |
| `url` | STRING | Request URL (new) |
| `api_type` | STRING | API type (new) |

## Column Mapping

| V1 Column | → | V2 Column | Transformation |
|-----------|---|-----------|----------------|
| `databricks_request_id` | → | `request_id` | Renamed |
| `client_request_id` | → | `request_tags["client_request_id"]` | Moved to map |
| `request_date` | → | *(dropped)* | Can be derived from event_time |
| `request_time` | → | `event_time` | Renamed |
| `status_code` | → | `status_code` | No change |
| `sampling_fraction` | → | `sampling_fraction` | No change |
| `execution_duration_ms` | → | `latency_ms` | Renamed |
| `request` | → | `request` | No change |
| `response` | → | `response` | No change |
| `served_entity_id` | → | `destination_id` | Renamed |
| `logging_error_codes` | → | `logging_error_codes` | No change |
| `requester` | → | `requester` | No change |
| *(new)* | | `time_to_first_byte_ms` | Set to NULL |
| *(new)* | | `schema_version` | Set to "v2_migrated" |
| *(new)* | | `url` | Set to NULL |
| *(new)* | | `api_type` | Set to NULL |

---
## Configuration

Update these values to match your source and target tables.

In [0]:
# Source: AI Gateway V1 inference table
SOURCE_CATALOG = "users"
SOURCE_SCHEMA = "yasamin_tari"
SOURCE_TABLE = "serving_inference_payload"  # Your V1 table name

# Target: Unity AI Gateway V2 inference table
TARGET_CATALOG = "users"
TARGET_SCHEMA = "yasamin_tari"
TARGET_TABLE = "yasamin-gateway-fallback-test_payload"  # Your V2 table name

# Construct full table names
source_table_name = f"{SOURCE_CATALOG}.{SOURCE_SCHEMA}.{SOURCE_TABLE}"
target_table_name = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.`{TARGET_TABLE}`"

# Temp view name for the transformed data (used in MERGE)
TEMP_VIEW_NAME = "v1_to_v2_transformed_data"

print(f"Source (V1): {source_table_name}")
print(f"Target (V2): {target_table_name}")

Source (V1): users.yasamin_tari.serving_inference_payload
Target (V2): users.yasamin_tari.`yasamin-gateway-fallback-test_payload`


---
## Step 1: Inspect Source Table

First, let's examine the source V1 table to understand its structure and volume.

In [0]:
# Read source V1 table
df_v1 = spark.read.table(source_table_name)

print("Source V1 Schema:")
df_v1.printSchema()

Source V1 Schema:
root
 |-- databricks_request_id: string (nullable = true)
 |-- request_date: date (nullable = true)
 |-- client_request_id: string (nullable = true)
 |-- request_time: timestamp (nullable = true)
 |-- status_code: integer (nullable = true)
 |-- sampling_fraction: double (nullable = true)
 |-- execution_duration_ms: long (nullable = true)
 |-- request: string (nullable = true)
 |-- response: string (nullable = true)
 |-- logging_error_codes: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- served_entity_id: string (nullable = true)
 |-- requester: string (nullable = true)



In [0]:
# Check record count
source_count = df_v1.count()
print(f"Source record count: {source_count:,}")

Source record count: 15


---
## Step 2: Inspect Target Table

Verify the target V2 table exists and examine its schema.

In [0]:
# Read target V2 table schema
df_v2_target = spark.read.table(target_table_name)

print("Target V2 Schema:")
df_v2_target.printSchema()

Target V2 Schema:
root
 |-- event_time: timestamp (nullable = true)
 |-- request_id: string (nullable = true)
 |-- request_tags: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)
 |-- status_code: integer (nullable = true)
 |-- sampling_fraction: double (nullable = true)
 |-- latency_ms: long (nullable = true)
 |-- request: string (nullable = true)
 |-- response: string (nullable = true)
 |-- destination_id: string (nullable = true)
 |-- logging_error_codes: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- requester: string (nullable = true)
 |-- time_to_first_byte_ms: long (nullable = true)
 |-- schema_version: string (nullable = true)
 |-- url: string (nullable = true)
 |-- api_type: string (nullable = true)



In [0]:
# Get target column order (we'll need this for the final write)
target_columns = [field.name for field in df_v2_target.schema.fields]

In [0]:
# Check existing record count in target
target_count_before = df_v2_target.count()
print(f"Target record count (before migration): {target_count_before:,}")

Target record count (before migration): 2


---
## Step 3: Define Migration Transformation

This function transforms V1 records to V2 schema.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, LongType, ArrayType

def migrate_v1_to_v2(df_v1):
    """
    Transform AI Gateway V1 inference table to Unity AI Gateway V2 schema.
    
    Key transformations:
    - databricks_request_id → request_id
    - client_request_id → request_tags map
    - request_time → event_time
    - execution_duration_ms → latency_ms
    - served_entity_id → destination_id
    - Add new V2 columns with NULL/default values
    """
    
    # Build request_tags map from client_request_id
    # This preserves the client_request_id in the new schema
    if "client_request_id" in df_v1.columns:
        request_tags_expr = F.when(
            F.col("client_request_id").isNotNull(),
            F.create_map(
                F.lit("client_request_id"), 
                F.col("client_request_id")
            )
        ).otherwise(
            F.create_map().cast("map<string,string>")
        )
    else:
        request_tags_expr = F.create_map().cast("map<string,string>")
    
    df_v2 = df_v1.select(
        # Renamed columns (V1 → V2)
        F.col("request_time").alias("event_time"),
        F.col("databricks_request_id").alias("request_id"),
        request_tags_expr.alias("request_tags"),
        
        # Unchanged columns
        F.col("status_code"),
        F.col("sampling_fraction"),
        
        # Renamed columns
        F.col("execution_duration_ms").alias("latency_ms"),
        
        # Unchanged columns
        F.col("request"),
        F.col("response"),
        
        # Renamed columns
        F.col("served_entity_id").alias("destination_id"),
        
        # Unchanged columns
        F.col("logging_error_codes"),
        F.col("requester"),
        
        # New V2 columns (set to NULL or identifier)
        F.lit(None).cast(LongType()).alias("time_to_first_byte_ms"),
        F.lit("v2_migrated").alias("schema_version"),  # Marks migrated records
        F.lit(None).cast(StringType()).alias("url"),
        F.lit(None).cast(StringType()).alias("api_type")
    )
    
    return df_v2

---
## Step 4: Apply Transformation

Transform the source data to V2 schema and preview the results.

In [0]:
# Apply the V1 → V2 transformation
df_v2 = migrate_v1_to_v2(df_v1)

In [0]:
# Reorder columns to match target table schema exactly
df_v2_ordered = df_v2.select(*target_columns)

print("Final column order matches target:")
print(df_v2_ordered.columns)

Final column order matches target:
['event_time', 'request_id', 'request_tags', 'status_code', 'sampling_fraction', 'latency_ms', 'request', 'response', 'destination_id', 'logging_error_codes', 'requester', 'time_to_first_byte_ms', 'schema_version', 'url', 'api_type']


---
## Step 5: Validate Before Writing

Perform validation checks before committing the migration.

In [0]:
# Validation checks
print("=" * 50)
print("PRE-MIGRATION VALIDATION")
print("=" * 50)

# Check 1: Record counts
v2_count = df_v2_ordered.count()
print(f"\n✓ Source records:      {source_count:,}")
print(f"✓ Transformed records: {v2_count:,}")
assert source_count == v2_count, "Record count mismatch!"
print("  → Counts match ✓")

# Check 2: No null request_ids (required for MERGE key)
null_ids = df_v2_ordered.filter(F.col("request_id").isNull()).count()
print(f"\n✓ Null request_ids: {null_ids}")
if null_ids > 0:
    print("  ⚠️ WARNING: Null request_ids will not be matched in MERGE")

# Check 3: No null event_times (required for MERGE key)
null_times = df_v2_ordered.filter(F.col("event_time").isNull()).count()
print(f"✓ Null event_times: {null_times}")
if null_times > 0:
    print("  ⚠️ WARNING: Null event_times will not be matched in MERGE")

# Check 4: Schema compatibility
source_cols = set(df_v2_ordered.columns)
target_cols = set(target_columns)
missing = target_cols - source_cols
extra = source_cols - target_cols
print(f"\n✓ Schema check:")
print(f"  Missing columns: {missing if missing else 'None'}")
print(f"  Extra columns:   {extra if extra else 'None'}")

print("\n" + "=" * 50)
print("VALIDATION COMPLETE - Ready for MERGE")
print("=" * 50)

PRE-MIGRATION VALIDATION

✓ Source records:      15
✓ Transformed records: 15
  → Counts match ✓

✓ Null request_ids: 0
✓ Null event_times: 0

✓ Schema check:
  Missing columns: None
  Extra columns:   None

VALIDATION COMPLETE - Ready for MERGE


---
## Step 6: Create Temp View for MERGE

Register the transformed DataFrame as a temp view so we can use it in SQL MERGE.

In [0]:
# Register transformed data as temp view for SQL MERGE
df_v2_ordered.createOrReplaceTempView(TEMP_VIEW_NAME)



---
## Step 7: Execute MERGE INTO

⚠️ **CAUTION**: The cell below will merge data into your target table. Review the validation results above before proceeding.


```sql
MERGE INTO target_table AS target
USING source_view AS source
ON target.request_id = source.request_id 
   AND target.event_time = source.event_time
WHEN NOT MATCHED THEN INSERT *
```

- **Match condition**: Uses `request_id` + `event_time` as composite key
- **NOT MATCHED**: Only inserts records that don't already exist
- **Idempotent**: Safe to run multiple times without creating duplicates

In [0]:
# Build the MERGE INTO statement
merge_sql = f"""
MERGE INTO {target_table_name} AS target
USING {TEMP_VIEW_NAME} AS source
ON target.request_id = source.request_id 
   AND target.event_time = source.event_time
WHEN NOT MATCHED THEN 
    INSERT (
        event_time,
        request_id,
        request_tags,
        status_code,
        sampling_fraction,
        latency_ms,
        request,
        response,
        destination_id,
        logging_error_codes,
        requester,
        time_to_first_byte_ms,
        schema_version,
        url,
        api_type
    )
    VALUES (
        source.event_time,
        source.request_id,
        source.request_tags,
        source.status_code,
        source.sampling_fraction,
        source.latency_ms,
        source.request,
        source.response,
        source.destination_id,
        source.logging_error_codes,
        source.requester,
        source.time_to_first_byte_ms,
        source.schema_version,
        source.url,
        source.api_type
    )
"""


In [0]:
# ============================================================
# EXECUTE MERGE
# ============================================================

result = spark.sql(merge_sql)
display(result)
print("\n✓ MERGE complete!")

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
15,0,0,15



✓ MERGE complete!


---
## Step 8: Post-Migration Validation

After running the MERGE, verify the migration was successful.

In [0]:
# Check record counts after MERGE
# Run this after uncommenting the MERGE above

post_merge_query = f"""
SELECT 
    schema_version,
    COUNT(*) as record_count,
    MIN(event_time) as earliest_event,
    MAX(event_time) as latest_event
FROM {target_table_name}
GROUP BY schema_version
ORDER BY schema_version
"""

display(spark.sql(post_merge_query))

schema_version,record_count,earliest_event,latest_event
1.0,2,2026-04-27T19:02:57.299Z,2026-04-27T19:03:34.474Z
v2_migrated,15,2026-05-12T15:45:48.806Z,2026-05-12T16:05:10.069Z


In [0]:
# Verify total count increased correctly
# Run this after the MERGE

target_count_after = spark.read.table(target_table_name).count()
records_inserted = target_count_after - target_count_before

print(f"Target count before MERGE: {target_count_before:,}")
print(f"Target count after MERGE:  {target_count_after:,}")
print(f"Records inserted:          {records_inserted:,}")
print(f"Source records:            {source_count:,}")

if records_inserted == source_count:
    print("\n✓ All source records were inserted (first run)")
elif records_inserted == 0:
    print("\n✓ No new records inserted (already migrated - idempotent!)")
else:
    print(f"\n⚠️ Partial insert: {records_inserted} of {source_count} records were new")

Target count before MERGE: 2
Target count after MERGE:  17
Records inserted:          15
Source records:            15

✓ All source records were inserted (first run)


---
## Appendix: Batch Migration with MERGE for Large Tables

For tables with millions of records, use date-based batching with MERGE to avoid memory issues while maintaining idempotency.

In [0]:
def migrate_in_batches_with_merge(source_table, target_table, date_column="request_date"):
    """
    Migrate large tables in date-based batches using MERGE INTO.
    
    This approach:
    1. Gets all distinct dates from source
    2. Processes one date at a time
    3. Uses MERGE INTO for each batch (idempotent)
    
    Benefits:
    - Avoids OOM errors on large tables
    - Provides progress visibility
    - Safe to resume if interrupted (MERGE is idempotent)
    - No duplicates even if re-run
    """
    
    # Get distinct dates
    dates = (spark.read.table(source_table)
             .select(F.col(date_column))
             .distinct()
             .orderBy(date_column)
             .collect())
    
    total_dates = len(dates)
    total_records = 0
    
    print(f"Found {total_dates} distinct dates to migrate")
    print("=" * 50)
    
    for i, row in enumerate(dates, 1):
        date_val = row[date_column]
        
        # Read batch for this date
        df_batch = (spark.read.table(source_table)
                    .filter(F.col(date_column) == date_val))
        
        batch_count = df_batch.count()
        total_records += batch_count
        
        print(f"[{i}/{total_dates}] {date_val}: {batch_count:,} records")
        
        if batch_count > 0:
            # Transform to V2
            df_batch_v2 = migrate_v1_to_v2(df_batch)
            
            # Reorder columns
            target_cols = [f.name for f in spark.read.table(target_table).schema.fields]
            df_batch_v2_ordered = df_batch_v2.select(*target_cols)
            
            # Create temp view for this batch
            batch_view = f"batch_migration_{i}"
            df_batch_v2_ordered.createOrReplaceTempView(batch_view)
            
            # Build MERGE statement for this batch
            batch_merge_sql = f"""
            MERGE INTO {target_table} AS target
            USING {batch_view} AS source
            ON target.request_id = source.request_id 
               AND target.event_time = source.event_time
            WHEN NOT MATCHED THEN INSERT *
            """
            
            # Execute MERGE (uncomment to run)
            # spark.sql(batch_merge_sql)
            print(f"         → Would MERGE {batch_count:,} records")
    
    print("=" * 50)
    print(f"Total records to migrate: {total_records:,}")
    return total_records

# Uncomment to run batch migration with MERGE:
# migrate_in_batches_with_merge(source_table_name, target_table_name)

---
## Summary

This notebook migrates inference table data from AI Gateway V1 to Unity AI Gateway V2 schema using **MERGE INTO** for idempotent operation.

**Key points:**
- Uses `MERGE INTO` instead of `APPEND` — safe to re-run without duplicates
- Match key: `request_id` + `event_time`
- Migrated records are marked with `schema_version = 'v2_migrated'` for tracking
- New V2 columns (`time_to_first_byte_ms`, `url`, `api_type`) are set to NULL for migrated data
- The `client_request_id` is preserved in the `request_tags` map
- Use batch migration with MERGE for large tables (>1M records)

**After migration:**
1. Verify record counts match
2. Spot-check sample records
3. Update any downstream queries to use V2 column names
4. Re-run safely if needed — MERGE is idempotent